<a href="https://colab.research.google.com/github/imagra93/ML-course-labs/blob/main/machine_learning/lab%20-%20Logistic%20Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Logistic Regression from Scratch

**Theory:** [notes_ML/02_logistic_regression.ipynb](../notes_ML/02_logistic_regression.ipynb) (sigmoid, log-odds, cross-entropy = MLE, gradient, convexity, metrics). This lab only implements it.

The model outputs the probability of class 1:

$$a = \sigma(z) = \frac{1}{1+e^{-z}}, \qquad z = \boldsymbol{\theta}^\top\mathbf{x} = \theta_0 + \theta_1 x_1 + \theta_2 x_2$$

and we predict class 1 when $a > 0.5$.

### Learning Objectives
- Implement the sigmoid and the binary cross-entropy loss
- Implement batch gradient descent
- Evaluate a binary classifier with accuracy, confusion matrix, and ROC/AUC
- Compare our implementation to scikit-learn

<div align='center'>

![Logistic Regression](https://upload.wikimedia.org/wikipedia/commons/6/6d/Exam_pass_logistic_curve.jpeg)

*Logistic regression fits an S-shaped sigmoid curve to binary outcome data. Unlike linear regression, predictions are always in $[0,1]$ and interpretable as probabilities.*

</div>

In [ ]:
%matplotlib inline
from IPython.display import HTML
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')

#### Generate the data
Two clouds of 2D points (`make_blobs`), one per class. Each point has 2 features $(x_1, x_2)$ and a label $y \in \{0, 1\}$.

In [ ]:
from sklearn.datasets import make_blobs
X, y = make_blobs(n_samples=200, centers=2,
                  random_state=10, cluster_std=3)

In [ ]:
m, n = X.shape

print(f"Num features is {n}")
print(f"Num samples is {m}")
print(f"y shape: {y.shape}  — classes: {sorted(set(y.tolist()))}")

# we work with y as a column vector of shape (m, 1), like X @ theta
y = y[:, None]
print(f"y shape after reshaping: {y.shape}")

Let's plot the data.

In [ ]:
passed = (y == 1).reshape(-1, 1)
failed = (y == 0).reshape(-1, 1)

fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(x=X[passed[:, 0], 0], y=X[passed[:, 0], 1],
                marker="^", color="tab:orange", s=70, label="Class 1 (Passed)", ax=ax)
sns.scatterplot(x=X[failed[:, 0], 0], y=X[failed[:, 0], 1],
                marker="X", color="tab:blue", s=70, label="Class 0 (Failed)", ax=ax)
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.set_title("Dataset: Binary Classification")
ax.legend()
plt.tight_layout()
plt.show()

## Feature Scaling / Standardisation

Before running gradient descent, we **standardise** each feature so it has mean 0 and standard deviation 1 (see notes 01, section 6, for why this makes gradient descent faster):

$$\tilde{x}_j = \frac{x_j - \mu_j}{\sigma_j}$$

We keep $\mu$ and $\sigma$: new data must be transformed with these **same** training values.

In [ ]:
def standarizeX(X):
    """
    Standarizes the features of X to have 0 mean and unit variance.

    Arguments:
    X -- feature matrix X (m, n)

    Returns:
    X -- standarized matix X
    mu, sigma -- mean and standard deviation of the features (for new samples)
    """

    mu = np.mean(X, axis=0)       # one mean per feature (column)
    sigma = np.std(X, axis=0)     # one std per feature (column)

    # standarize X: subtract the mean and divide by the std (column by column, via broadcasting)
    X = (X - mu) / sigma

    return X, mu, sigma

In [ ]:
X, mu, sigma = standarizeX(X)

In [ ]:
print("-" * 45)
print("  Feature Standardisation")
print("-" * 45)
print(f"  Mean  : {mu.round(4)}")
print(f"  Std   : {sigma.round(4)}")
print("-" * 45)

In [ ]:
print("First 5 rows of X:")
print(X[:5])

Before continuing, we will add the first column of all 1's ($x_0$ for the intercept term).

In [ ]:
X = np.concatenate([np.ones((m,1)), X], axis = 1)

In [ ]:
print("First 5 rows of X:")
print(X[:5])

## Helper functions

Now we will define a few helper functions that we will put together to build the logistic regression model.

### Sigmoid (Logistic) Function

$$\sigma(z) = \frac{1}{1+e^{-z}}$$

It squashes any number into $(0, 1)$, so its output can be read as a probability. `np.exp` works element-wise, so the same code works for a number or for a whole array.

In [ ]:
def sigmoid(z):
    """
    Implement the sigmoid function

    Arguments:
    z -- a scalar (float) - or numpy array.

    Return:
    A -- the sigmoid function evaluated on z  - or numpy array.
    """

    # Sigmoid function.
    A = 1 / (1 + np.exp(-z))

    return A

In [ ]:
xs_sigmoid = np.linspace(-10, 10, 300)
ys_sigmoid = sigmoid(xs_sigmoid)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(xs_sigmoid, ys_sigmoid, color='steelblue', linewidth=2.5,
        label=r'$\sigma(z) = 1 / (1 + e^{-z})$')
ax.axhline(0.5, color='crimson', linestyle='--', linewidth=1.2, label=r'$\sigma(0) = 0.5$')
ax.axhline(1.0, color='gray',    linestyle=':',  linewidth=1.0, label='Asymptotes (0, 1)')
ax.axhline(0.0, color='gray',    linestyle=':',  linewidth=1.0)
ax.axvline(0.0, color='crimson', linestyle='--', linewidth=0.8, alpha=0.5)
ax.set_xlabel('$z$')
ax.set_ylabel(r'$\sigma(z)$')
ax.set_title('Sigmoid (Logistic) Function')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

You should verify that the output is bounded between 0 and 1, and equals exactly 0.5 at $z=0$. Let's check a few key values:

In [ ]:
print("Sigmoid sanity checks:")
print(f"  sigma(0)   = {sigmoid(0):.4f}   (expected 0.5000)")
print(f"  sigma(10)  = {sigmoid(10):.6f} (expected ≈ 1.0000)")
print(f"  sigma(-10) = {sigmoid(-10):.6f} (expected ≈ 0.0000)")
print(f"  sigma(-z) = 1 - sigma(z): sigma(-2) = {sigmoid(-2):.4f}, 1-sigma(2) = {1-sigmoid(2):.4f}")

### Initialise the parameters

One $\theta$ for the bias plus one per feature, stored as a column vector of shape $(n+1, 1)$. Starting at zeros is fine: the cost is convex, so the start does not change the final answer.

In [ ]:
def initialize(n):
    """
    Initialise the thetas to tensors of dimensions (n+1,1)

    Arguments:
    n -- a scalar (int)

    Return:
    theta -- a matrix of dimensions (n+1,1) containing all zero
    """
    theta = np.zeros((n + 1, 1))

    return theta

In [ ]:
theta = initialize(n)
print(f"Initialised theta — shape: {theta.shape}  (bias + {n} features, all zeros)")

## Cost Function: Binary Cross-Entropy

$$\mathcal{J}(\boldsymbol{\theta}) = -\frac{1}{m}\sum_{i=1}^m \left[ y^{(i)}\log a^{(i)} + (1-y^{(i)})\log(1-a^{(i)}) \right], \qquad a^{(i)} = \sigma(\boldsymbol{\theta}^\top\mathbf{x}^{(i)})$$

**Intuition:** only one of the two terms is active for each example.
- If $y=1$: the loss is $-\log a$, huge when the model says $a \approx 0$.
- If $y=0$: the loss is $-\log(1-a)$, huge when the model says $a \approx 1$.

In code: **1)** all probabilities at once $\mathbf{a} = \sigma(\mathbf{X}\boldsymbol{\theta})$, **2)** the loss of every example, **3)** the average. We add a tiny $\epsilon$ inside the logs so we never compute $\log 0$.

In [ ]:
def cost_function(X, y, theta):
    """
    cost_function(X, y, theta) computes the cost of using theta as the
    parameter for logistic regression to fit the data points in X and y.

    Arguments:
    theta -- weight vector, a numpy array of size (n+1, 1)
    X -- data of shape (m, n+1)
    y -- true "label" vector (containing 0 for class 0, 1 for class 1), of shape (m,1)

    Output:
    J -- the cost function.
    """
    ## number of training examples
    m = len(y)

    ## 1) Calculate the estimates A (probability of class 1 for every example).
    A = sigmoid(np.matmul(X, theta))

    ## 2) Loss of every example, 3) average. Add epsilon for numerical stability.
    epsilon = 1e-7
    losses = -(y * np.log(A + epsilon) + (1 - y) * np.log(1 - A + epsilon))
    J = np.mean(losses)

    return J

In [ ]:
cost = cost_function(X, y, theta)
print(f"Initial cost (zero weights ≈ random classifier): {cost:.4f}  (expected ≈ ln 2 ≈ 0.6931)")

## Gradient of the Cost Function

(Derivation in section 5 of the notes.) The result has the same form as for linear regression:

$$\nabla_{\boldsymbol{\theta}}\mathcal{J}(\boldsymbol{\theta}) = \frac{1}{m}\mathbf{X}^\top(\mathbf{a} - \mathbf{y})$$

The only difference is that the prediction is $\mathbf{a} = \sigma(\mathbf{X}\boldsymbol{\theta})$ instead of $\mathbf{X}\boldsymbol{\theta}$.

In code: **1)** probabilities $\mathbf{a}$, **2)** errors $\mathbf{a} - \mathbf{y}$, **3)** multiply by $\mathbf{X}^\top$ and divide by $m$.

In [ ]:
def calculate_gradient(X, y, theta):
    """
    calculate_gradient(X, y, theta) computes the gradient of the cost function for
    the current thetas given X and y.

    Arguments:
    theta -- weight vector, a numpy array of size (n+1, 1)
    X -- data of shape (m, n+1)
    y -- true "label" vector (containing 0 for class 0, 1 for class 1), of shape (m,1)

    output:
    dJ -- Gradient of the cost function with respect to the thetas (n+1,1)
    """
    ## number of training examples
    m = len(y)

    ## 1) Calculate the estimates A.
    A = sigmoid(np.matmul(X, theta))

    ## 2) errors and 3) gradient (vectorized)
    errors = A - y
    dJ = X.T @ errors / m

    return dJ

In [ ]:
dJ = calculate_gradient(X, y, theta)
print(f"Gradient shape : {dJ.shape}")
for j, g in enumerate(dJ.flatten()):
    print(f"  dJ/dtheta_{j} = {g:.6f}")

## Gradient descent
With these functions we perform the gradient descent algorithm: at every iteration, compute the gradient and take a small step against it.

$$\boldsymbol{\theta} \leftarrow \boldsymbol{\theta} - \alpha\, \nabla_{\boldsymbol{\theta}}\mathcal{J}(\boldsymbol{\theta})$$

In [ ]:
def optimize(theta, X, y, num_iterations, learning_rate, print_cost = True):
    """
    This function optimizes theta by running a gradient descent algorithm

    Arguments:
    theta -- weight vector, a numpy array of size (n+1, 1)
    X -- data of shape (m, n+1)
    y -- true "label" vector (containing 0 for class 0, 1 for class 1), of shape (m,1)
    num_iterations -- number of iterations of the optimization loop
    learning_rate -- learning rate of the gradient descent update rule
    print_cost -- True to print the loss every 200 steps

    Returns:
    theta -- a vector containing the weights
    costs -- list of all the costs computed during the optimization, this will be used to plot the learning curve.
    """
    costs = []

    for i in range(num_iterations):

        cost = cost_function(X, y, theta)
        dJ = calculate_gradient(X, y, theta)

        theta = theta - learning_rate * dJ

        if i % 10 == 0:
            costs.append(cost)

        if print_cost and i % 200 == 0:
            print(f"  [{i:5d}/{num_iterations}]  Cost = {cost:.6f}")

    return theta, costs

Set the training parameters:

In [ ]:
num_iterations = 5000
learning_rate = 0.005

In [ ]:
optimized_theta, costs = optimize(theta, X, y, num_iterations, learning_rate)

In [ ]:
print("-" * 45)
print("  Gradient Descent — Optimised Weights")
print("-" * 45)
for j, t in enumerate(optimized_theta.flatten()):
    label = "bias" if j == 0 else f"feat. {j}"
    print(f"  theta_{j} ({label:7s}) : {t:.6f}")
print("-" * 45)

## Cost function vs. number of iterations

Let's see how the cost function decrease with the number of iterations. We have the value of the cost function every $10$ iterations so we can easily plot it.

In [ ]:
final_cost = costs[-1]
iters = list(range(0, len(costs) * 10, 10))

plt.figure(figsize=(8, 5))
plt.plot(iters, costs, color='steelblue', linewidth=2, label='Training loss')
plt.axhline(final_cost, color='crimson', linestyle='--', linewidth=1.2,
            label=f'Final cost = {final_cost:.4f}')
plt.annotate(f'{final_cost:.4f}',
             xy=(iters[-1], final_cost),
             xytext=(-65, 15), textcoords='offset points',
             color='crimson', fontsize=9,
             arrowprops=dict(arrowstyle='->', color='crimson', lw=1.0))
plt.xlabel('Iteration')
plt.ylabel('Binary Cross-Entropy Loss')
plt.title('Cost Convergence — Logistic Regression')
plt.legend()
plt.tight_layout()
plt.show()

## Prediction

The trained model outputs $a = \sigma(\boldsymbol{\theta}^\top\mathbf{x}) \in (0,1)$, the estimated probability that $y=1$. We turn it into a class with the threshold $0.5$:

$$\hat{y} = \begin{cases} 1 & \text{if } a > 0.5 \\ 0 & \text{otherwise} \end{cases}$$

Since $\sigma(z) > 0.5 \iff z > 0$, this is the same as predicting 1 when $\boldsymbol{\theta}^\top\mathbf{x} > 0$. (Other thresholds: notes, section 9.2.)

In [ ]:
def predict(theta, X):
    '''
    Predict whether the label is 0 or 1

    Arguments:
    theta -- weights, a numpy array of size (n+1, 1)
    X -- data of size (m, n+1)

    Returns:
    Y_prediction -- a numpy array (vector) containing all predictions (0/1)
    '''

    m  = X.shape[0]
    Y_prediction = np.zeros((m,1))

    A = sigmoid(np.matmul(X, theta))

    Y_prediction[A>0.5] = 1

    assert (Y_prediction.shape == (m,1))

    return Y_prediction


In [ ]:
pred = predict(optimized_theta, X)

And the score (accuracy) is:

In [ ]:
acc = np.sum(pred == y) / m
print(f"Accuracy: {acc:.4f}  ({acc*100:.2f}%)")

### Decision Boundary (From-Scratch Model)

Since we have 2 features, we can visualise the linear decision boundary learned by our gradient descent model. The boundary is where $\boldsymbol{\theta}^\top\mathbf{x} = 0$, i.e., $\theta_0 + \theta_1\tilde{x}_1 + \theta_2\tilde{x}_2 = 0$, which gives:

$$\tilde{x}_2 = -\frac{\theta_1}{\theta_2}\tilde{x}_1 - \frac{\theta_0}{\theta_2}$$

In [ ]:
theta0, theta1, theta2 = optimized_theta.flatten()
slope_scratch = -theta1 / theta2
intercept_scratch = -theta0 / theta2

xmin, xmax, ymin, ymax = -3.0, 3.0, -3.0, 3.0
xd = np.array([xmin, xmax])
yd = slope_scratch * xd + intercept_scratch

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(xd, yd, 'k--', lw=1.8, label='Decision boundary (scratch)')
ax.fill_between(xd, yd, ymin, color='tab:blue',   alpha=0.12, label='Class 0 region')
ax.fill_between(xd, yd, ymax, color='tab:orange', alpha=0.12, label='Class 1 region')
ax.scatter(X[passed[:, 0], 1], X[passed[:, 0], 2],
           s=45, alpha=0.75, color='tab:orange', edgecolors='white', label='Class 1 (Passed)')
ax.scatter(X[failed[:, 0], 1], X[failed[:, 0], 2],
           s=45, alpha=0.75, color='tab:blue',   edgecolors='white', label='Class 0 (Failed)')
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.set_xlabel(r'$\tilde{x}_1$ (standardised)')
ax.set_ylabel(r'$\tilde{x}_2$ (standardised)')
ax.set_title('Decision Boundary — From-Scratch Gradient Descent')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### Predicted Probability Surface

We can visualise the probability $P(y=1|\mathbf{x})$ over the feature space as a heatmap:

In [ ]:
xx, yy = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-3, 3, 200))
grid = np.c_[np.ones(xx.ravel().shape), xx.ravel(), yy.ravel()]
proba = sigmoid(grid @ optimized_theta).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7, 6))
cf = ax.contourf(xx, yy, proba, levels=50, cmap='RdYlBu_r', alpha=0.8)
plt.colorbar(cf, ax=ax, label='P(y=1 | x)')
ax.contour(xx, yy, proba, levels=[0.5], colors='black', linewidths=2, linestyles='--')
ax.scatter(X[passed[:, 0], 1], X[passed[:, 0], 2],
           s=45, alpha=0.85, color='tab:orange', edgecolors='white', label='Class 1')
ax.scatter(X[failed[:, 0], 1], X[failed[:, 0], 2],
           s=45, alpha=0.85, color='tab:blue',   edgecolors='white', label='Class 0')
ax.set_xlabel(r'$\tilde{x}_1$ (standardised)')
ax.set_ylabel(r'$\tilde{x}_2$ (standardised)')
ax.set_title(r'Predicted Probability Surface — $P(y=1 \mid \mathbf{x})$')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Evaluation Metrics

Accuracy alone can be misleading (e.g. with imbalanced classes). We also look at the confusion matrix, precision / recall / $F_1$ and the ROC curve with its AUC. Definitions and interpretation are in section 9 of the notes.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, roc_auc_score

print("=== Confusion Matrix ===")
print(confusion_matrix(y.flatten(), pred.flatten()))

print("\n=== Classification Report ===")
print(classification_report(y.flatten(), pred.flatten(), target_names=['Class 0', 'Class 1']))

# ROC Curve
A_proba = sigmoid(np.matmul(X, optimized_theta)).flatten()
fpr, tpr, thresholds = roc_curve(y.flatten(), A_proba)
auc = roc_auc_score(y.flatten(), A_proba)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion matrix heatmap
cm = confusion_matrix(y.flatten(), pred.flatten())
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred 0','Pred 1'], yticklabels=['True 0','True 1'])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# ROC curve
axes[1].plot(fpr, tpr, 'b-', linewidth=2, label=f'Logistic Reg (AUC = {auc:.3f})')
axes[1].plot([0,1], [0,1], 'k--', label='Random classifier (AUC = 0.5)')
axes[1].fill_between(fpr, tpr, alpha=0.1)
axes[1].set_xlabel('False Positive Rate (FPR)')
axes[1].set_ylabel('True Positive Rate (TPR / Recall)')
axes[1].set_title('ROC Curve')
axes[1].legend()
plt.tight_layout(); plt.show()
print(f"\nAUC = {auc:.4f}  (1.0 = perfect, 0.5 = random)")

## Bonus 1: Logistic Regression with scikit-learn

Everything we wrote by hand is available in one class, `sklearn.linear_model.LogisticRegression`. Using it takes three steps, the same for **every** sklearn model:

1. **Create** the model and choose its hyperparameters: `model = LogisticRegression(...)`
2. **Fit** it to the training data: `model.fit(X, y)`
3. **Use** it: `model.predict(X)`, `model.predict_proba(X)`, `model.score(X, y)`

This is how our functions map to sklearn:

| Our code | sklearn |
|---|---|
| `initialize` + `optimize` | `model.fit(X, y)` |
| `optimized_theta[0]` (bias) | `model.intercept_` |
| `optimized_theta[1:]` (weights) | `model.coef_` |
| `sigmoid(X @ theta)` | `model.predict_proba(X)[:, 1]` |
| `predict(theta, X)` | `model.predict(X)` |
| `np.sum(pred == y) / m` | `model.score(X, y)` |

### Step 1: create the model

The most important hyperparameters (here written with their default values):

- `C=1.0`: sklearn **regularises by default**. It adds an L2 penalty that keeps $\boldsymbol{\theta}$ small, and `C` is the *inverse* of its strength ($C = 1/\lambda$). Small `C` means strong regularisation; very large `C` means almost none. (Regularisation: notes 03.)
- `fit_intercept=True`: sklearn adds its own bias $\theta_0$, so we must **not** pass the column of ones.
- `solver='lbfgs'`: instead of plain gradient descent it uses a smarter method that also uses the curvature of the cost, so it needs far fewer iterations and no learning rate.
- `max_iter=100`: maximum number of iterations of the solver.

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(C=1.0, fit_intercept=True, solver='lbfgs', max_iter=100)

### Step 2: fit

We pass the standardised features **without** the column of ones (`X[:, 1:]`), and the labels as a flat vector of shape `(m,)` (`y.ravel()`), which is what sklearn expects.

In [ ]:
X_sk = X[:, 1:]          # drop our bias column: sklearn adds its own intercept
y_sk = y.ravel()         # shape (m, 1) -> (m,)

logreg.fit(X_sk, y_sk)

### Step 3: inspect the learned parameters

After `fit`, the parameters are stored in `intercept_` (our $\theta_0$) and `coef_` (our $\theta_1, \theta_2$). Let's put them next to ours.

In [ ]:
theta_sk = np.concatenate([logreg.intercept_, logreg.coef_.ravel()])

print(f"{'':10s} {'ours (GD)':>12s} {'sklearn':>12s}")
for j, name in enumerate(['theta_0', 'theta_1', 'theta_2']):
    print(f"{name:10s} {optimized_theta[j, 0]:12.4f} {theta_sk[j]:12.4f}")

The signs and the rough proportions agree, but the numbers are not the same. Why?

- Our classes are **(almost) perfectly separable**. In that case the cost can always be lowered a bit more by making $\boldsymbol{\theta}$ bigger (the sigmoid becomes steeper and more confident), so the unregularised minimum is at infinity (notes 02, section 8).
- **Our gradient descent** stopped after 5000 iterations while the cost was still going down, so it stopped "on the way".
- **sklearn** stopped where the L2 penalty (`C=1`) balances the pull towards larger $\boldsymbol{\theta}$.

We can see this by changing `C`: the less regularisation (larger `C`), the larger the coefficients, while the accuracy barely changes.

In [ ]:
print(f"{'C':>8s} {'theta_0':>9s} {'theta_1':>9s} {'theta_2':>9s} {'accuracy':>9s}")
for C in [0.01, 0.1, 1, 10, 100, 10000]:
    model = LogisticRegression(C=C).fit(X_sk, y_sk)
    t = np.concatenate([model.intercept_, model.coef_.ravel()])
    print(f"{C:8g} {t[0]:9.3f} {t[1]:9.3f} {t[2]:9.3f} {model.score(X_sk, y_sk):9.3f}")

### Step 4: predict

- `predict_proba(X)` returns one column per class: `[:, 0]` is $P(y=0)$ and `[:, 1]` is $P(y=1)$. Each row sums to 1.
- `predict(X)` applies the 0.5 threshold and returns the class.
- `score(X, y)` returns the accuracy: $\frac{\text{number of correct predictions}}{m}$.

Notice that sklearn's probabilities are more extreme (closer to 0 or 1) than ours: its $\boldsymbol{\theta}$ is larger, so $z = \boldsymbol{\theta}^\top\mathbf{x}$ is larger and the sigmoid is steeper. The predicted **classes** are almost the same.

In [ ]:
proba_sk = logreg.predict_proba(X_sk)
print("First 5 rows of predict_proba  [P(y=0), P(y=1)]:")
print(proba_sk[:5].round(4))
print("\nOur model, P(y=1) for the same rows:")
print(sigmoid(X[:5] @ optimized_theta).ravel().round(4))

pred_sk = logreg.predict(X_sk)
print(f"\nFirst 10 predicted classes: {pred_sk[:10]}")
print(f"First 10 true classes     : {y_sk[:10]}")

score = logreg.score(X_sk, y_sk)
print(f"\nsklearn training accuracy: {score:.4f}  ({score*100:.2f}%)")
print(f"our     training accuracy: {np.mean(pred.ravel() == y_sk):.4f}")

### Step 5: predict a new data point

A new point arrives in the **original** units. Before giving it to either model we must apply the **same** standardisation as in training, using the saved $\mu$ and $\sigma$ (never statistics computed on the new data). For our model we also add the 1 for the bias; sklearn does not need it.

In [ ]:
x_new = np.array([[4.0, 5.0]])                 # raw features (x_1, x_2)
x_new_std = (x_new - mu) / sigma               # same transformation as the training data

# our model: add the bias column, then predict
x_new_ours = np.concatenate([np.ones((1, 1)), x_new_std], axis=1)
p_ours = sigmoid(x_new_ours @ optimized_theta)[0, 0]

# sklearn: no bias column
p_sk = logreg.predict_proba(x_new_std)[0, 1]

print(f"New point {x_new[0]} -> standardised {x_new_std[0].round(3)}")
print(f"  ours   : P(y=1) = {p_ours:.4f} -> class {int(p_ours > 0.5)}")
print(f"  sklearn: P(y=1) = {p_sk:.4f} -> class {logreg.predict(x_new_std)[0]}")

### Step 6: compare the decision boundaries

With two features the boundary $\theta_0 + \theta_1\tilde{x}_1 + \theta_2\tilde{x}_2 = 0$ is a line: $\tilde{x}_2 = -\frac{\theta_1}{\theta_2}\tilde{x}_1 - \frac{\theta_0}{\theta_2}$. We draw ours and sklearn's together.

In [ ]:
b_sk   = logreg.intercept_[0]
w1_sk, w2_sk = logreg.coef_.ravel()
slope_sk = -w1_sk / w2_sk
c_sk     = -b_sk  / w2_sk

xmin, xmax, ymin, ymax = -3.0, 3.0, -3.0, 3.0
xd = np.array([xmin, xmax])

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(xd, slope_scratch * xd + intercept_scratch, 'k--', lw=1.8, label='Ours (gradient descent)')
ax.plot(xd, slope_sk * xd + c_sk, color='crimson', lw=1.8, label='sklearn')
ax.scatter(X[passed[:, 0], 1], X[passed[:, 0], 2],
           s=45, alpha=0.75, color='tab:orange', edgecolors='white', label='Class 1 (Passed)')
ax.scatter(X[failed[:, 0], 1], X[failed[:, 0], 2],
           s=45, alpha=0.75, color='tab:blue',   edgecolors='white', label='Class 0 (Failed)')
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.set_xlabel(r'$\tilde{x}_1$ (standardised)')
ax.set_ylabel(r'$\tilde{x}_2$ (standardised)')
ax.set_title('Decision Boundaries — ours vs. sklearn')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

Both lines separate the classes almost equally well. They differ slightly because the two $\boldsymbol{\theta}$'s are different points on the way to the (infinite) unregularised solution. The boundary depends only on the **ratios** of the $\theta$'s, not their size.

In practice you would use sklearn: fewer lines, a faster and more robust solver, and regularisation built in. Writing it from scratch shows what `fit` is doing underneath.